# 📊 Notebook 01: FIRR/WACC Recomputation — Namma Metro Phase 2A/2B
**Paper:** P1 — T-InvIT Bengaluru RWA Tokenization  
**Author:** Sandeep S | PhD Scholar, University of Mysore  
**Data Source:** ADB Financial Analysis (ind-53326-001-fa.pdf)  
**Integrity Note:** All base data extracted from ADB source. Model extensions clearly labeled.

In [ ]:
import numpy as np
import numpy_financial as npf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print('Data collection timestamp:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Source: ADB Financial Analysis — ind-53326-001-fa.pdf')
print('URL: https://www.adb.org/sites/default/files/linked-documents/ind-53326-001-fa.pdf')

## Step 1: Load BMRCL Financial Data
> **TODO:** Run DataAgent_ADB.py first to populate data/raw/BMRCL/  
> Then load: `data/processed/BMRCL_financials_clean.csv`

In [ ]:
# ADB-reported baseline values (Source: ind-53326-001-fa.pdf, Table FA-1)
# CITATION: Asian Development Bank. (2024). Financial Analysis: Bangalore Metro Rail Project Phase 2.
# ADB Project Number: 53326-001. Retrieved from https://www.adb.org/sites/default/files/linked-documents/ind-53326-001-fa.pdf

ADB_FIRR = 0.01921   # 1.921% — POST-TAX FIRR from ADB Table FA-1
ADB_WACC = 0.02629   # 2.629% — WACC in real terms from ADB Table FA-2
WACC_SPREAD = ADB_WACC - ADB_FIRR  # Viability gap

print(f'ADB Reported FIRR: {ADB_FIRR*100:.3f}%')
print(f'ADB Reported WACC: {ADB_WACC*100:.3f}%')
print(f'Viability Gap (WACC - FIRR): {WACC_SPREAD*100:.3f}%')
print('NOTE: Negative spread means project destroys capital without sovereign subsidy.')

## Step 2: Sensitivity Analysis
Test how FIRR changes with ±20% variation in ridership and cost

In [ ]:
# SENSITIVITY TABLE — MODEL SIMULATION (not primary data)
# Assumptions clearly stated for reproducibility

scenarios = {
    'Base Case (ADB)':   {'ridership_factor': 1.00, 'cost_factor': 1.00},
    'Ridership +20%':    {'ridership_factor': 1.20, 'cost_factor': 1.00},
    'Ridership -20%':    {'ridership_factor': 0.80, 'cost_factor': 1.00},
    'Cost +20%':         {'ridership_factor': 1.00, 'cost_factor': 1.20},
    'Cost -20%':         {'ridership_factor': 1.00, 'cost_factor': 0.80},
    'T-InvIT Scenario':  {'ridership_factor': 1.10, 'cost_factor': 0.90},  # Tokenization impact
}

results = []
for name, s in scenarios.items():
    # Simplified IRR approximation based on ADB multipliers
    adjusted_firr = ADB_FIRR * s['ridership_factor'] / s['cost_factor']
    viability = 'Viable' if adjusted_firr > ADB_WACC else 'Not Viable (Needs Subsidy)'
    results.append({'Scenario': name, 'FIRR (%)': f'{adjusted_firr*100:.3f}', 'Viability': viability})

df_sensitivity = pd.DataFrame(results)
print(df_sensitivity.to_string(index=False))
df_sensitivity.to_csv('data/processed/FIRR_sensitivity_table.csv', index=False)
print('\n✅ Saved to data/processed/FIRR_sensitivity_table.csv')